In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# Load Data
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')

# Feature Engineering
def create_features(df):
    df = df.copy()
    
    # Fix column names if needed
    if 'Max HR' in df.columns:
        df['Max heart rate'] = df['Max HR']
    if 'BP' in df.columns:
        df['Resting blood pressure'] = df['BP']

    df['Cardiac_Workload'] = df['Age'] * df['Max heart rate']
    df['Stress_Factor'] = df['ST depression'] * df['Slope of ST']
    df['Chol_Age_Ratio'] = df['Cholesterol'] / df['Age']
    
    df['High_BP'] = (df['Resting blood pressure'] > 130).astype(int)
    df['High_Chol'] = (df['Cholesterol'] > 240).astype(int)
    df['Total_Risk_Factors'] = df['High_BP'] + df['High_Chol'] + df['FBS over 120']
    
    return df

train = create_features(train)
test = create_features(test)

train['Heart Disease'] = train['Heart Disease'].map({'Presence': 1, 'Absence': 0})

cat_cols = ['Chest pain type', 'EKG results', 'Slope of ST', 'Thallium', 
            'High_BP', 'High_Chol', 'Total_Risk_Factors']

for col in cat_cols:
    if col in train.columns:
        train[col] = train[col].astype('category')
        test[col] = test[col].astype('category')

X = train.drop(['id', 'Heart Disease'], axis=1)
y = train['Heart Disease']
X_test = test.drop('id', axis=1)

kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(
        n_estimators=3000,
        learning_rate=0.01,
        max_depth=5,
        subsample=0.6,
        colsample_bytree=0.5,
        min_child_weight=3,
        eval_metric='auc',
        enable_categorical=True,
        tree_method='hist',
        early_stopping_rounds=150,
        random_state=42,
        n_jobs=-1,
        device='cuda'
    )
    
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    val_pred = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_pred
    test_preds += model.predict_proba(X_test)[:, 1] / 10
    
    score = roc_auc_score(y_val, val_pred)
    cv_scores.append(score)
    print(f"Fold {fold+1} AUC: {score:.5f}")

print(f"\nOverall CV AUC: {roc_auc_score(y, oof_preds):.5f}")
print(f"Average Fold AUC: {np.mean(cv_scores):.5f}")

submission = pd.DataFrame({'id': test['id'], 'Heart Disease': test_preds})
submission.to_csv('submission.csv', index=False)
print("Saved: submission.csv")